# Datenanalyse mit SQL & Python - Tag 3: Übungen

**Teil A:** 30-Minuten Warm-up & Recap zu Tag 2 mit `household_budget`  
**Teil B:** Pandas-Übungen mit einer bereits verbundenen Shop-Tabelle (`shop_joined`)  
**Mini-Projekt:** Shop-Daten als Gruppenarbeit: KPIs auswählen, 1-2 Diagramme erstellen, 3 Insights und eine Empfehlung formulieren  
**Ziel:** Erst SQL-Konzepte aus Tag 2 wiederholen, dann mit Pandas Daten prüfen, filtern, gruppieren, visualisieren und als Business-Empfehlung präsentieren.

# 09:00–09:30 | Tag-2-Refresh mit 'Household Budget'

## Einrichtung & Daten laden

Führe diese Zelle zuerst aus. Sie lädt:

- `household_budget` als SQL-Tabelle für Teil A
- `shop_joined` als Pandas-DataFrame für Teil B und das Mini-Projekt
- `fitness_joined` als Pandas-DataFrame für die gemeinsame Übung im Hauptnotebook

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

SHOP_DIR = 'https://raw.githubusercontent.com/chiaoya/Data_to_Decision_with_SQL_Python/refs/heads/main/course_data/shop/'
FITNESS_DIR = 'https://raw.githubusercontent.com/chiaoya/Data_to_Decision_with_SQL_Python/refs/heads/main/course_data/fitness/'
BUDGET_PATH = 'https://raw.githubusercontent.com/chiaoya/Data_to_Decision_with_SQL_Python/refs/heads/main/course_data/household_budget.csv'


budget_df = pd.read_csv(BUDGET_PATH)
budget_df['date'] = pd.to_datetime(budget_df['date'])
budget_df['month'] = budget_df['date'].dt.strftime('%Y-%m')

orders = pd.read_csv(SHOP_DIR + 'shop_orders.csv')
customers = pd.read_csv(SHOP_DIR + 'shop_customers.csv')
products = pd.read_csv(SHOP_DIR + 'shop_products.csv')

bookings = pd.read_csv(FITNESS_DIR + 'fitness_bookings.csv')
members = pd.read_csv(FITNESS_DIR + 'fitness_members.csv')
classes = pd.read_csv(FITNESS_DIR + 'fitness_classes.csv')

shop_joined = (
    orders
    .merge(customers, on='customer_id', how='left')
    .merge(products, on='product_id', how='left')
)
shop_joined['revenue'] = shop_joined['quantity'] * shop_joined['price']

fitness_joined = (
    bookings
    .merge(members, on='member_id', how='left')
    .merge(classes, on='class_id', how='left')
)
fitness_joined['revenue'] = fitness_joined['price']

conn = sqlite3.connect(':memory:')
budget_df.to_sql('budget', conn, index=False, if_exists='replace')

print('SQL-Tabelle budget:', budget_df.shape)
print('Pandas-DataFrame shop_joined:', shop_joined.shape)
print('Pandas-DataFrame fitness_joined:', fitness_joined.shape)


SQL-Tabelle budget: (60, 6)
Pandas-DataFrame shop_joined: (50, 11)
Pandas-DataFrame fitness_joined: (50, 9)


## Teil A: 30-Minuten Warm-up & Recap zu Tag 2 mit Household Budget

In Teil A arbeitest du mit SQL und der Tabelle `budget`.

Diese Tabelle enthält:

- `date`
- `month`
- `category`
- `description`
- `amount`
- `type`

**Ziel:** Wiederhole die wichtigsten Konzepte aus Tag 2:

- `GROUP BY`
- `HAVING`
- Subquery
- CTE mit `WITH`
- Interpretation von aggregierten Ergebnissen


### Übung A1: Tabelle verstehen (ca. 3 Minuten)

**Aufgabe:** Zeige die ersten 10 Zeilen aus `budget` an.


In [ ]:
# Aufgabe: Erste Zeilen anzeigen.
query = '''
SELECT *
FROM budget
LIMIT _____;
'''

result = pd.read_sql_query(query, conn)
result


### Übung A2: GROUP BY nach Kategorie (ca. 5 Minuten)

**Aufgabe:** Berechne die gesamten Ausgaben pro Kategorie.

**Hinweis:** Filtere zuerst auf `type = 'Expense'` und gruppiere nach `category`.


In [ ]:
# Aufgabe: Ausgaben pro Kategorie berechnen.
query = '''
SELECT
    category,
    ABS(SUM(amount)) AS ausgaben
FROM budget
WHERE type = '_____'
GROUP BY _____
ORDER BY ausgaben DESC;
'''

result = pd.read_sql_query(query, conn)
result


### Übung A3: HAVING verwenden (ca. 5 Minuten)

**Aufgabe:** Zeige nur Kategorien mit mehr als 600 Euro Ausgaben.

**Hinweis:** Die Bedingung bezieht sich auf `SUM(amount)`, also brauchst du `HAVING`.


In [ ]:
# Aufgabe: Gruppen mit HAVING filtern.
query = '''
SELECT
    category,
    ABS(SUM(amount)) AS ausgaben
FROM budget
WHERE type = 'Expense'
GROUP BY category
HAVING _____ > 600
ORDER BY ausgaben DESC;
'''

result = pd.read_sql_query(query, conn)
result


### Übung A4: Monatsanalyse mit GROUP BY (ca. 5 Minuten)

**Aufgabe:** Berechne Ausgaben, Einnahmen und Netto-Betrag pro Monat.

**Hinweis:** `amount` ist bei Ausgaben negativ und bei Einnahmen positiv.


In [ ]:
# Aufgabe: Monatswerte berechnen.
query = '''
SELECT
    month,
    ABS(SUM(CASE WHEN type = 'Expense' THEN amount ELSE 0 END)) AS ausgaben,
    SUM(CASE WHEN type = 'Income' THEN amount ELSE 0 END) AS einnahmen,
    SUM(amount) AS netto
FROM budget
GROUP BY _____
ORDER BY month;
'''

result = pd.read_sql_query(query, conn)
result


### Übung A5: Subquery (ca. 6 Minuten)

**Aufgabe:** Welche Kategorien haben höhere Ausgaben als der Durchschnitt aller Kategorien?

**Hinweis:** Die innere Abfrage berechnet zuerst die Ausgaben pro Kategorie.


In [2]:
# Aufgabe: Kategorien über dem Durchschnitt finden.
query = '''
SELECT
    category,
    ABS(SUM(amount)) AS ausgaben
FROM budget
WHERE type = 'Expense'
GROUP BY category
HAVING ABS(SUM(amount)) > (
    SELECT AVG(kategorie_ausgaben)
    FROM (
        SELECT ABS(SUM(amount)) AS kategorie_ausgaben
        FROM budget
        WHERE type = 'Expense'
        GROUP BY category
    )
)
ORDER BY ausgaben DESC;
'''

result = pd.read_sql_query(query, conn)
result


,category,ausgaben
0,Dining,2237
1,Entertainment,1812
2,Rent,1352


### Übung A6: CTE als Vergleich (ca. 6 Minuten)

**Aufgabe:** Löse A5 noch einmal mit einer CTE.

**Vergleich:** Welche Version ist leichter zu lesen: Subquery oder CTE?


In [3]:
# Aufgabe: Gleiche Logik mit WITH schreiben.
query = '''
WITH category_expenses AS (
    SELECT
        category,
        ABS(SUM(amount)) AS ausgaben
    FROM budget
    WHERE type = 'Expense'
    GROUP BY category
),
average_expense AS (
    SELECT AVG(ausgaben) AS durchschnitt
    FROM category_expenses
)
SELECT
    ce.category,
    ce.ausgaben
FROM category_expenses AS ce
CROSS JOIN average_expense AS ae
WHERE ce.ausgaben > ae.durchschnitt
ORDER BY ce.ausgaben DESC;
'''

result = pd.read_sql_query(query, conn)
result


,category,ausgaben
0,Dining,2237
1,Entertainment,1812
2,Rent,1352


### Reflexion Teil A (ca. 3 Minuten)

Diskutiert kurz:

1. Wann brauchst du `HAVING` statt `WHERE`?
1. Warum kann eine CTE lesbarer sein als eine Subquery?
1. Welche Kategorie oder welcher Monat wäre aus Budget-Sicht kritisch?


# 14:20–15:40 | Praxis mit Shop-Daten & Mini-Projekt

## Teil B: Shop-Daten mit Pandas

Du arbeitest mit `shop_joined`. Diese Tabelle enthält Bestellungen, Kundendaten und Produktdaten in einem DataFrame.

Wichtige Spalten:

- `order_id`
- `customer_id`
- `name`
- `city`
- `product_name`
- `category`
- `quantity`
- `price`
- `revenue`


### Übung B1: DataFrame verstehen

**Aufgabe:** Zeige die ersten Zeilen, die Spaltennamen und die Datentypen von `shop_joined` an.


In [4]:
# Aufgabe: Erste Orientierung im DataFrame.
# shop_joined

,order_id,customer_id,product_id,quantity,name,city,age,product_name,category,price,revenue
0,1001,14,120,4,Paul Neumann,Hamburg,48,Water Bottle,Sports,33,132
1,1002,20,110,3,Tim Krause,Cologne,26,Training Gloves,Sports,50,150
2,1003,4,105,4,Noah Weber,Berlin,25,Kitchen Towel Set,Home,13,52
3,1004,12,101,4,Elias Klein,Berlin,27,Running Shoes,Sports,137,548
4,1005,19,108,3,Laura Lange,Cologne,30,Mountain Backpack,Sports,146,438
5,1006,2,107,2,Lukas Schneider,Cologne,28,Fitness Tracker,Sports,101,202
6,1007,8,110,2,Finn Schulz,Berlin,29,Training Gloves,Sports,50,100
7,1008,2,111,4,Lukas Schneider,Cologne,28,Air Purifier,Home,154,616
8,1009,17,108,2,Marie Braun,Berlin,55,Mountain Backpack,Sports,146,292
9,1010,13,105,1,Lina Wolf,Berlin,39,Kitchen Towel Set,Home,13,13


In [ ]:
# Aufgabe: Erste Orientierung im DataFrame.
shop_joined._____()


In [ ]:
# Aufgabe: Spaltennamen anzeigen.
shop_joined._____


In [ ]:
# Aufgabe: Datentypen und fehlende Werte prüfen.
shop_joined._____()


### Übung B2: Filtern

**Aufgabe:** Filtere alle Bestellungen aus der Stadt `Berlin`.


In [ ]:
# Aufgabe: Filtere Bestellungen aus Berlin.
berlin_orders = shop_joined[shop_joined['_____'] == 'Berlin']
berlin_orders.head()


### Übung B3: Neue Kennzahl prüfen

**Aufgabe:** Zeige `quantity`, `price` und `revenue`. Prüfe, ob `revenue = quantity * price` sinnvoll berechnet wurde.


In [ ]:
# Aufgabe: Relevante Spalten auswählen.
shop_joined[['_____', '_____', '_____']].head(10)


### Übung B4: Umsatz pro Stadt

**Aufgabe:** Berechne den gesamten Umsatz pro Stadt.

**Hinweis:** Nutze `groupby`, wähle `revenue` und summiere.


In [ ]:
# Aufgabe: Umsatz pro Stadt berechnen.
revenue_by_city = (
    shop_joined
    .groupby('_____')['_____']
    .sum()
    .sort_values(ascending=False)
)
revenue_by_city


### Übung B5: Umsatz pro Produktkategorie

**Aufgabe:** Welche Produktkategorie bringt den höchsten Umsatz?


In [ ]:
# Aufgabe: Umsatz pro Kategorie berechnen.
revenue_by_category = (
    shop_joined
    .groupby('_____')['revenue']
    .sum()
    .sort_values(ascending=False)
)
revenue_by_category


### Übung B6: Top 5 Bestellungen

**Aufgabe:** Zeige die fünf Bestellungen mit dem höchsten Umsatz.


In [ ]:
# Aufgabe: Nach revenue sortieren und Top 5 anzeigen.
top_orders = shop_joined.sort_values(by='_____', ascending=False).head(_____)
top_orders[['order_id', 'name', 'city', 'product_name', 'quantity', 'price', 'revenue']]


### Übung B7: Visualisierung

**Aufgabe:** Erstelle ein Balkendiagramm für den Umsatz pro Produktkategorie.


In [ ]:
# Aufgabe: Balkendiagramm mit Seaborn erstellen.
category_plot_data = revenue_by_category.reset_index()
category_plot_data.columns = ['category', 'revenue']

sns.barplot(data=category_plot_data, x='revenue', y='category')
plt.title('Umsatz pro Produktkategorie')
plt.xlabel('Umsatz')
plt.ylabel('Kategorie')
plt.show()


### Reflexion Teil B

Diskutiert kurz:

1. Welche Stadt ist im Shop-Datensatz besonders wichtig?
1. Welche Produktkategorie ist umsatzstark?
1. Welche zusätzliche Information wäre für eine bessere Shop-Analyse hilfreich?


## Mini-Projekt: Shop-Daten als Gruppenarbeit

Ihr nutzt jetzt `shop_joined`, also denselben Shop-Datensatz wie in Teil B.

**Ziel:** Entwickelt aus bekannten Analysen eine kleine Business-Präsentation.

Erwartetes Ergebnis pro Gruppe:

- 1-2 Diagramme
- 3 zentrale Insights
- 1 konkrete Empfehlung für das Shop-Team
- 3-4 Minuten Präsentation pro Gruppe

### Aufgabe 1) Welche Stadt ist kommerziell am wichtigsten?

**Diagramm:** Balkendiagramm

**Fragen:**

- Welche Stadt erzielt den höchsten Umsatz?
- Ist die Stadt auch bei der Anzahl der Bestellungen führend?
- Welche KPI ist hier aussagekräftiger: `revenue`, `quantity` oder `order count`?

**Insight (Beispiel):**

- Eine Stadt kann viel Umsatz bringen, obwohl sie nicht die meisten Bestellungen hat. Das deutet auf größere oder teurere Warenkörbe hin.

In [ ]:
# Aufgabe 1: Umsatz, Menge und Bestellungen pro Stadt analysieren
city_kpis = (
    shop_joined
    .groupby('city')
    .agg(
        revenue=('revenue', 'sum'),
        quantity=('quantity', 'sum'),
        orders=('order_id', 'count')
    )
    .sort_values('revenue', ascending=False)
)

city_kpis

### Aufgabe 2) Welche Produktkategorie treibt den Umsatz?

**Diagramm:** Balkendiagramm

**Fragen:**

- Welche Kategorie bringt den höchsten Umsatz?
- Welche Kategorie verkauft die meisten Stückzahlen?
- Gibt es Kategorien mit hoher Menge, aber niedrigerem Umsatz?

**Insight (Beispiel):**

- Eine Kategorie mit hoher Menge, aber niedrigerem Umsatz könnte preisgetrieben sein. Eine Kategorie mit hohem Umsatz trotz niedriger Menge könnte strategisch besonders wertvoll sein.

In [ ]:
# Aufgabe 2: Kategorie-KPIs berechnen
category_kpis = (
    shop_joined
    .groupby('category')
    .agg(
        revenue=('revenue', 'sum'),
        quantity=('quantity', 'sum'),
        avg_price=('price', 'mean')
    )
    .sort_values('revenue', ascending=False)
)

category_kpis

### Aufgabe 3) Welche Produkte sind besonders relevant?

**Diagramm:** Top-5-Balkendiagramm

**Fragen:**

- Welche fünf Produkte bringen den höchsten Umsatz?
- Sind das auch die Produkte mit der höchsten verkauften Menge?
- Welche Produkte sollte das Shop-Team besonders beobachten?

**Insight (Beispiel):**

- Top-Produkte nach Umsatz sind gute Kandidaten für Lagerbestand, Kampagnen oder Bundles.

In [ ]:
# Aufgabe 3: Top-Produkte nach Umsatz
product_kpis = (
    shop_joined
    .groupby('product_name')
    .agg(
        revenue=('revenue', 'sum'),
        quantity=('quantity', 'sum'),
        avg_price=('price', 'mean')
    )
    .sort_values('revenue', ascending=False)
)

product_kpis.head(5)

### Aufgabe 4) Welche Kundengruppen sind interessant?

**Diagramm:** Balkendiagramm oder Boxplot

**Fragen:**

- Unterscheiden sich jüngere und ältere Kundinnen/Kunden im Kaufverhalten?
- Welche Altersgruppe erzielt mehr Umsatz oder kauft größere Mengen?
- Welche Segmentierung wäre für Marketing sinnvoll?

**Insight (Beispiel):**

- Wenn eine Altersgruppe mehr Umsatz erzeugt, kann das Shop-Team Angebote, Kommunikation oder Produktempfehlungen gezielter ausrichten.

In [ ]:
# Aufgabe 4: Altersgruppen bilden und vergleichen
shop_segmented = shop_joined.copy()
shop_segmented['age_group'] = pd.cut(
    shop_segmented['age'],
    bins=[0, 29, 39, 49, 120],
    labels=['<30', '30-39', '40-49', '50+']
)

age_group_kpis = (
    shop_segmented
    .groupby('age_group', observed=True)
    .agg(
        revenue=('revenue', 'sum'),
        quantity=('quantity', 'sum'),
        orders=('order_id', 'count')
    )
    .sort_values('revenue', ascending=False)
)

age_group_kpis

### Aufgabe 5) Empfehlung ableiten

**Output:** 3 Insights + 1 Empfehlung

**Fragen:**

- Welche KPI ist für eure Empfehlung am wichtigsten?
- Welche Stadt, Kategorie, Produktgruppe oder Kundengruppe sollte priorisiert werden?
- Welche Unsicherheit oder Einschränkung hat eure Analyse?

**Hilfssatz für eure Präsentation:**

- Ich sehe ...
- Das deutet darauf hin, dass ...
- Deshalb empfehlen wir ...

**Empfehlung (Beispiel):**

- Deshalb empfehlen wir, die umsatzstärkste Kategorie in den Top-Städten stärker zu bewerben und gleichzeitig Produkte mit hoher Menge, aber niedrigem Umsatz auf Preisstrategie oder Bundle-Potenzial zu prüfen.

In [ ]:
# Aufgabe 5: Präsentationsgerüst ausfüllen
presentation = {
    'Insight 1': '...',
    'Insight 2': '...',
    'Insight 3': '...',
    'Empfehlung': '...',
    'Einschränkung': '...'
}

for key, value in presentation.items():
    print(f'{key}: {value}')

### Präsentations-Checkliste

Vor der Präsentation:

- Wählt 1-2 Diagramme aus.
- Formuliert 3 Insights in ganzen Sätzen.
- Verbindet jeden Insight mit einer KPI.
- Schließt mit einer konkreten Empfehlung.
- Nennt eine Einschränkung, z. B. kleiner Datensatz oder fehlende Zeitdimension.

## Abschluss

Heute verbindest du drei Denkweisen:

- Teil A wiederholt SQL-Konzepte aus Tag 2.
- Teil B überträgt Analysefragen auf Pandas mit Shop-Daten.
- Das Mini-Projekt zeigt, wie aus KPIs, Diagrammen und Insights eine kurze Business-Empfehlung wird.

Der wichtigste Schritt ist nicht nur der Code, sondern die Frage: Welche Entscheidung kann ich mit dem Ergebnis besser treffen?

## Musterlösungen

Die folgenden Lösungen zeigen jeweils ein mögliches sauberes Muster. Bei Visualisierungen und Interpretationen sind andere gute Varianten möglich.


### Lösungen Teil A: SQL-Recap

In [ ]:
sql_loesungen = {
    'A1': 'SELECT * FROM budget LIMIT 10;',
    'A2': '''
        SELECT category, SUM(amount) AS ausgaben
        FROM budget
        WHERE type = 'Expense'
        GROUP BY category
        ORDER BY ausgaben DESC;
    ''',
    'A3': '''
        SELECT category, SUM(amount) AS ausgaben
        FROM budget
        WHERE type = 'Expense'
        GROUP BY category
        HAVING SUM(amount) > 100
        ORDER BY ausgaben DESC;
    ''',
    'A4': '''
        SELECT month, SUM(amount) AS betrag
        FROM budget
        GROUP BY month
        ORDER BY month;
    ''',
    'A5': '''
        SELECT category, SUM(amount) AS ausgaben
        FROM budget
        WHERE type = 'Expense'
        GROUP BY category
        HAVING SUM(amount) > (
            SELECT AVG(category_total)
            FROM (
                SELECT SUM(amount) AS category_total
                FROM budget
                WHERE type = 'Expense'
                GROUP BY category
            )
        )
        ORDER BY ausgaben DESC;
    ''',
    'A6': '''
        WITH category_totals AS (
            SELECT category, SUM(amount) AS ausgaben
            FROM budget
            WHERE type = 'Expense'
            GROUP BY category
        ),
        average_total AS (
            SELECT AVG(ausgaben) AS avg_ausgaben
            FROM category_totals
        )
        SELECT category, ausgaben
        FROM category_totals
        CROSS JOIN average_total
        WHERE ausgaben > avg_ausgaben
        ORDER BY ausgaben DESC;
    ''',
}

for name, query in sql_loesungen.items():
    print(f'--- {name} ---')
    display(pd.read_sql_query(query, conn).head(10))


### Lösungen Teil B: Shop-Daten mit Pandas

In [ ]:
# B1
shop_joined.head()
shop_joined.columns
shop_joined.info()

# B2
berlin_orders = shop_joined.loc[shop_joined['city'] == 'Berlin']
berlin_orders.head()

# B3
shop_joined[['order_id', 'customer_id', 'city', 'product_name', 'quantity', 'price', 'revenue']].head()

# B4
revenue_by_city = (
    shop_joined
    .groupby('city')['revenue']
    .sum()
    .sort_values(ascending=False)
)
revenue_by_city

# B5
revenue_by_category = (
    shop_joined
    .groupby('category')['revenue']
    .sum()
    .sort_values(ascending=False)
)
revenue_by_category

# B6
top_orders = shop_joined.sort_values('revenue', ascending=False).head(5)
top_orders


In [ ]:
# B7
plot_data = revenue_by_category.reset_index()
plot_data.columns = ['category', 'revenue']

sns.barplot(data=plot_data, x='revenue', y='category')
plt.title('Umsatz pro Produktkategorie')
plt.xlabel('Umsatz')
plt.ylabel('Kategorie')
plt.show()


### Mini-Projekt: mögliche KPI-Bausteine

Diese Bausteine sind keine vollständige Musterlösung. Sie helfen euch, eure eigene Präsentation aufzubauen.

In [ ]:
# KPI-Bausteine für das Shop Mini-Projekt
city_kpis = shop_joined.groupby('city').agg(
    revenue=('revenue', 'sum'),
    quantity=('quantity', 'sum'),
    orders=('order_id', 'count')
).sort_values('revenue', ascending=False)

category_kpis = shop_joined.groupby('category').agg(
    revenue=('revenue', 'sum'),
    quantity=('quantity', 'sum'),
    avg_price=('price', 'mean')
).sort_values('revenue', ascending=False)

product_kpis = shop_joined.groupby('product_name').agg(
    revenue=('revenue', 'sum'),
    quantity=('quantity', 'sum'),
    avg_price=('price', 'mean')
).sort_values('revenue', ascending=False)

city_kpis, category_kpis, product_kpis.head(5)